In [15]:
pip install escher

In [16]:
pip install "cobra[all]"


In [0]:
from __future__ import print_function
import escher
from escher import Builder
import cobra
from time import sleep
import cobra.test

In [0]:
#Function for Output solution
def  output_solution(solution):
     print('Growth Rate (1/h): ' + str(solution.get_primal_by_id("BIOMASS_Ec_iJO1366_WT_53p95M")))
     print('Lycopene Production Rate (mmol/gdcw/h): ' + str(solution.get_primal_by_id('LYCO-dem')))
     print('Lycopene Yield (mol/mol glucose): ' + str(-solution.get_primal_by_id('LYCO-dem')/solution.get_primal_by_id('EX_glc__D_e')))


### Suggested workflow

### 1. Set up a model for the wild type E. coli strain used in the paper.

In [0]:
#Load the model for genome scale E. coli iJO1366
model = cobra.test.create_test_model("ecoli")
#model.optimize

#Set constraints for aerobic growth in glucose minimal media
model.reactions.get_by_id( "EX_glc__D_e").lower_bound= -10
model.reactions.get_by_id( "EX_o2_e").lower_bound = -15

**Visualization**

In [20]:
builder = Builder()
solution = model.optimize()
builder.map_name = 'iJO1366.Central metabolism'
builder.model_name = 'e_coli_core'
builder.reaction_data = solution.fluxes
builder.metabolite_data = solution.shadow_prices
builder.hide_secondary_metabolites = True

In [0]:
builder.reaction_scale = [
    { 'type': 'min' , 'color' : '#000000' , 'size' : 12},
      { 'type': 'median' , 'color' : '#ffffff' , 'size' : 20},
        { 'type': 'max' , 'color' : '#ff0000' , 'size' : 25},
    ]
builder.reaction_scale_preset = 'GaBuRd'
builder.save_html("example_map.html")

In [22]:
print(model.reactions)

[<Reaction DM_4crsol_c at 0x7f74ba043e48>, <Reaction DM_5drib_c at 0x7f74ba043f28>, <Reaction DM_aacald_c at 0x7f74ba044240>, <Reaction DM_amob_c at 0x7f74ba044390>, <Reaction DM_mththf_c at 0x7f74ba044470>, <Reaction DM_oxam_c at 0x7f74ba0445c0>, <Reaction BIOMASS_Ec_iJO1366_WT_53p95M at 0x7f74ba0446a0>, <Reaction BIOMASS_Ec_iJO1366_core_53p95M at 0x7f74ba0447f0>, <Reaction EX_12ppd__R_e at 0x7f74ba04fcc0>, <Reaction EX_12ppd__S_e at 0x7f74ba044cf8>, <Reaction EX_14glucan_e at 0x7f74ba044cc0>, <Reaction EX_15dap_e at 0x7f74ba044dd8>, <Reaction EX_23camp_e at 0x7f74ba044eb8>, <Reaction EX_23ccmp_e at 0x7f74ba046278>, <Reaction EX_23cgmp_e at 0x7f74ba046048>, <Reaction EX_23cump_e at 0x7f74ba0461d0>, <Reaction EX_23dappa_e at 0x7f74ba0465c0>, <Reaction EX_26dap__M_e at 0x7f74ba046358>, <Reaction EX_2ddglcn_e at 0x7f74ba0465f8>, <Reaction EX_34dhpac_e at 0x7f74ba057940>, <Reaction EX_3amp_e at 0x7f74ba057a20>, <Reaction EX_3cmp_e at 0x7f74ba057b38>, <Reaction EX_3gmp_e at 0x7f74ba057c88>

### 2. Add the lycopene pathway to the model.

In [0]:
#Add crtEBI pathway for lycopene production
#Hint: see Alper et al 2005 Met Eng, Supp Info for reactions
#New metabolites: ggpp_c, phyto_c, lyco_c
from cobra import Metabolite
coa_c = model.metabolites.get_by_id( 'coa_c')
ipdp_c = model.metabolites.get_by_id( 'ipdp_c')
frdp_c = model.metabolites.get_by_id( 'frdp_c')
ppi_c = model.metabolites.get_by_id( 'ppi_c')
nadp_c = model.metabolites.get_by_id( 'nadp_c')
nadph_c = model.metabolites.get_by_id( 'nadph_c')

#Create new metabolites
ggpp_c = Metabolite( 'ggpp_c', formula='C20H36O7P2', name='Geranylgeranyl Pyrophospate', compartment ='c')
phyto_c = Metabolite( 'phyto_c', formula='C40H64', name='Phytoene', compartment ='c')
lyco_c = Metabolite( 'lyco_c', formula='C40H56', name='Lycopene', compartment ='c')

In [0]:
#New reactions: CRTE, CRTB, CRTI, LYCO-dem
from cobra import Reaction
#add CRTE:
reaction1 = Reaction('CRTE')
reaction1.name = 'Geranylgeranyl diphosphate (GGPP) synthase'
reaction1.subsystem = 'Lycopene biosynthesis'
reaction1.lower_bound = 0
reaction1.upper_bound = 1000
reaction1.add_metabolites({ipdp_c: -1.0, frdp_c: -1.0, ggpp_c: 1.0, ppi_c: 1.0})
model.add_reaction(reaction1)

#add CRTB:
reaction2 = Reaction('CRTB')
reaction2.name = 'Phytoene synthase'
reaction2.subsystem = 'Lycopene biosynthesis'
reaction2.lower_bound = 0
reaction2.upper_bound = 1000
reaction2.add_metabolites({ggpp_c: -2.0, phyto_c: 1.0, ppi_c: 1.0})
model.add_reaction(reaction2)

#add CRTI:
reaction3 = Reaction('CRTI')
reaction3.name = 'Phytoene desaturase'
reaction3.subsystem = 'Lycopene biosynthesis'
reaction3.lower_bound = 0
reaction3.upper_bound = 1000
reaction3.add_metabolites({phyto_c: -1.0, nadp_c: -8.0, lyco_c: 1.0, nadph_c: 8.0})
model.add_reaction(reaction3)

#add LYCO-dem:
reaction4 = Reaction('LYCO-dem')
reaction4.name = 'Lycopene demand'
reaction4.subsystem = 'Lycopene biosynthesis'
reaction4.lower_bound = 0
reaction4.upper_bound = 1000
reaction4.add_metabolites({lyco_c: -1.0})
model.add_reaction(reaction4)

In [0]:
#objective function
model.reactions.get_by_id('BIOMASS_Ec_iJO1366_core_53p95M').objective_coefficient = 0
model.reactions.get_by_id('BIOMASS_Ec_iJO1366_WT_53p95M').objective_coefficient = 1
model.reactions.get_by_id('LYCO-dem').objective_coefficient = 0.5

### 3. Simulate the model from Step 2

What objective function should you choose to answer the questions above?




In [0]:
# use for observing theoretical maximum yield of lycopene
model_1 = model.copy()

# use for knock_out+gene overexpression
model_2 = model.copy()

# use for knock_out
model_3 = model.copy()

# use for gene overexpression
model_4 = model.copy()

In [27]:
model_1.reactions.get_by_id('BIOMASS_Ec_iJO1366_core_53p95M').objective_coefficient = 0
model_1.reactions.get_by_id('BIOMASS_Ec_iJO1366_WT_53p95M').objective_coefficient = 1.0

#Solve
solution=model_1.optimize() #solution is stored at model.solution

#Output solution
output_solution(solution)

Growth Rate (1/h): 0.8168839340845325
Lycopene Production Rate (mmol/gdcw/h): 0.1909680042423488
Lycopene Yield (mol/mol glucose): 0.01909680042423488


Theoretical Maximum

In [28]:
#Set the objective to lycopene production
model_1.reactions.get_by_id('BIOMASS_Ec_iJO1366_core_53p95M').objective_coefficient = 0
model_1.reactions.get_by_id('BIOMASS_Ec_iJO1366_WT_53p95M').objective_coefficient = 0
model_1.reactions.get_by_id('LYCO-dem').objective_coefficient = 1.0

#Solve
solution = model_1.optimize()

#Output solution
output_solution(solution)

Growth Rate (1/h): 0.0
Lycopene Production Rate (mmol/gdcw/h): 1.1019165727170228
Lycopene Yield (mol/mol glucose): 0.11019165727170228


Wild type strain

In [29]:
model_1.reactions.get_by_id('BIOMASS_Ec_iJO1366_core_53p95M').objective_coefficient = 0
model_1.reactions.get_by_id('BIOMASS_Ec_iJO1366_WT_53p95M').objective_coefficient = 1
model_1.reactions.get_by_id('LYCO-dem').objective_coefficient = 0.5

#Solve
solution=model_1.optimize() #solution is stored at model.solution

#Output solution
output_solution(solution)

Growth Rate (1/h): 0.8168839340845268
Lycopene Production Rate (mmol/gdcw/h): 0.19096800424234786
Lycopene Yield (mol/mol glucose): 0.019096800424234787


### 4. Add gene knockouts from the paper to the lycopene model.


Which gene knockouts can we simulate? gdhA, aceE, ytjC(gmpB), fdhF 

In [0]:
#Knockout genes gdhA, aceE, ytjC(gpmB), fdhF  get From  2005a
model_3.genes.b1761.knock_out() # gdhA
model_3.genes.b0114.knock_out() # aceE
model_3.genes.b4395.knock_out() # ytjC
model_3.genes.b4079.knock_out() # fdhF

### 5. Simulate the model from Step 4.


In [31]:
solution_3 = model_3.optimize()
output_solution(solution_3)

Growth Rate (1/h): 0.6907866633676604
Lycopene Production Rate (mmol/gdcw/h): 0.2950734841554434
Lycopene Yield (mol/mol glucose): 0.029507348415544338


In [32]:
model_3.summary()

### 6. Add gene overexpression to the lycopene model from Step 2.


gene name : dxs, idi , ispFD (ispF and ispD) reactions: DXPS, IPDDI, MECDPS, MEPCT


In [33]:
from cobra.flux_analysis import flux_variability_analysis
reactions_OE = [model_4.reactions.DXPS, model_4.reactions.IPDDI, model_4.reactions.MECDPS, model_4.reactions.MEPCT]
fva = flux_variability_analysis(model_4, reaction_list = reactions_OE, fraction_of_optimum=0.9,loopless=False)
print (fva)

         minimum   maximum
DXPS    0.005807  3.412359
IPDDI  -2.559498  0.852591
MECDPS  0.005441  3.412088
MEPCT   0.005441  3.412088


In [0]:
# Overexpress by setting lower bound is greater than  upper bound
model_4.reactions.get_by_id('DXPS').lower_bound = 4
model_4.reactions.get_by_id('IPDDI').lower_bound = 1.4
model_4.reactions.get_by_id('MECDPS').lower_bound = 4
model_4.reactions.get_by_id('MEPCT').lower_bound = 4

### 7. Simulate the model from Step 6.


In [36]:
solution_4 = model_4.optimize()
output_solution(solution_4)

Growth Rate (1/h): 0.3636282726175124
Lycopene Production Rate (mmol/gdcw/h): 0.6998278220129155
Lycopene Yield (mol/mol glucose): 0.06998278220129155


In [37]:
model_4.summary()

### 8. Create a lycopene-producing model with overexpression + knockouts.


In [38]:
reactions_OE = [model_2.reactions.DXPS, model_2.reactions.IPDDI, model_2.reactions.MECDPS, model_2.reactions.MEPCT]
fva = flux_variability_analysis(model_2, reaction_list = reactions_OE, fraction_of_optimum=0.9)
print (fva)

         minimum   maximum
DXPS    0.005807  3.412359
IPDDI  -2.559498  0.852591
MECDPS  0.005441  3.412088
MEPCT   0.005441  3.412088


In [0]:
#Overexpress dxs, idi, ispFD
model_2.reactions.get_by_id('DXPS').lower_bound = 4
model_2.reactions.get_by_id('IPDDI').lower_bound = 1.6
model_2.reactions.get_by_id('MECDPS').lower_bound = 4
model_2.reactions.get_by_id('MEPCT').lower_bound = 4

model_2.genes.b1761.knock_out() # gdhA
model_2.genes.b0114.knock_out() # aceA
model_2.genes.b4395.knock_out() # ytjC
model_2.genes.b4079.knock_out() # fdhF

### 9. Simulate the model from Step 8.

In [40]:
solution_2 = model_2.optimize()
output_solution(solution_2)

Growth Rate (1/h): 0.2641504320879474
Lycopene Production Rate (mmol/gdcw/h): 0.7998749247704064
Lycopene Yield (mol/mol glucose): 0.07998749247704065


In [41]:
model_2.summary()